# Probability and Statistics for Machine Learning

**What is Probability?**  
Probability is how we quantify uncertainty. In ML, everything is uncertain — predictions, data, model outputs. Probability gives us the tools to reason about that uncertainty.

**Why does it matter?**  
- Naive Bayes is built entirely on probability
- Loss functions like cross entropy come from probability theory
- Logistic regression outputs a probability, not a class
- Understanding overfitting requires understanding distributions
- Bayesian ML, dropout, and uncertainty estimation all need this

**What you'll learn:**
- Basic probability rules
- Probability distributions
- Bayes theorem
- Statistics — mean, variance, covariance
- Central limit theorem
- Hypothesis testing
- Where all of this shows up in real ML

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## 1. Basic Probability

Probability of an event = number of favourable outcomes / total outcomes

P(A) is always between 0 and 1.  
P(A) = 0 means impossible. P(A) = 1 means certain.

In [ ]:
# Simulate a coin flip
np.random.seed(42)
flips = np.random.choice(['H', 'T'], size=1000)

counts = Counter(flips)
p_heads = counts['H'] / len(flips)
p_tails = counts['T'] / len(flips)

print('P(Heads):', p_heads)
print('P(Tails):', p_tails)
print('Sum:', p_heads + p_tails)    # always 1

# Complement rule: P(not A) = 1 - P(A)
print('P(not Heads):', 1 - p_heads)

In [ ]:
# Joint probability — P(A and B)
# If A and B are independent: P(A and B) = P(A) * P(B)

# Example: probability of getting heads twice in a row
p_heads = 0.5
p_two_heads = p_heads * p_heads
print('P(two heads in a row):', p_two_heads)    # 0.25

# Union rule — P(A or B) = P(A) + P(B) - P(A and B)
# Example: probability of rolling a 2 or an even number on a die
p_two   = 1/6
p_even  = 3/6       # {2, 4, 6}
p_both  = 1/6       # 2 is both
p_union = p_two + p_even - p_both
print('P(2 or even):', p_union)                 # 3/6 = 0.5

## 2. Conditional Probability

P(A | B) = probability of A given that B already happened

**Formula:** P(A | B) = P(A and B) / P(B)

This is the foundation of Bayes theorem and Naive Bayes classifier.

In [ ]:
# Real ML example: spam detection
# P(spam | contains word 'free') = ?

total_emails    = 1000
spam_emails     = 300
contains_free   = 150    # emails containing word 'free'
spam_with_free  = 120    # spam emails containing 'free'

p_spam              = spam_emails / total_emails
p_free              = contains_free / total_emails
p_free_given_spam   = spam_with_free / spam_emails
p_spam_given_free   = (spam_with_free / total_emails) / p_free

print('P(spam):', p_spam)
print('P(free):', p_free)
print('P(free | spam):', p_free_given_spam)
print('P(spam | free):', p_spam_given_free)

## 3. Bayes Theorem

**Formula:** P(A | B) = P(B | A) * P(A) / P(B)

In ML terms:
- P(A) = prior — what we believe before seeing data
- P(B | A) = likelihood — how likely is the data given our belief
- P(A | B) = posterior — updated belief after seeing data

This is the core of Naive Bayes classifier and Bayesian ML.

In [ ]:
# Medical test example — classic Bayes theorem
# Disease affects 1% of population
# Test is 99% accurate (true positive rate)
# Test has 5% false positive rate
# Question: if you test positive, what is the actual probability you have the disease?

p_disease       = 0.01      # prior — 1% have disease
p_no_disease    = 1 - p_disease
p_pos_given_disease     = 0.99  # true positive rate
p_pos_given_no_disease  = 0.05  # false positive rate

# Total probability of testing positive
p_positive = (p_pos_given_disease * p_disease +
              p_pos_given_no_disease * p_no_disease)

# Bayes theorem
p_disease_given_pos = (p_pos_given_disease * p_disease) / p_positive

print('P(positive test):', round(p_positive, 4))
print('P(disease | positive test):', round(p_disease_given_pos, 4))
print(f'\nOnly {p_disease_given_pos*100:.1f}% chance of actually having disease')
print('Even with a 99% accurate test — because disease is rare')
print('This is why ML models on imbalanced datasets need more than just accuracy')

## 4. Probability Distributions

A distribution describes how likely each value is. In ML, data follows distributions and understanding them helps you choose the right model.

In [ ]:
np.random.seed(42)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# 1. Uniform Distribution — equal probability for all values
# ML use: initialising weights, random sampling
uniform = np.random.uniform(0, 1, 1000)
axes[0,0].hist(uniform, bins=30, color='steelblue', alpha=0.7)
axes[0,0].set_title('Uniform Distribution')
axes[0,0].set_xlabel('ML use: weight initialisation')

# 2. Normal (Gaussian) Distribution — bell curve
# ML use: most natural data, weight initialisation, noise assumption
normal = np.random.normal(0, 1, 1000)
axes[0,1].hist(normal, bins=30, color='green', alpha=0.7)
axes[0,1].set_title('Normal Distribution')
axes[0,1].set_xlabel('ML use: most common assumption')

# 3. Bernoulli Distribution — binary outcomes (0 or 1)
# ML use: binary classification output
bernoulli = np.random.binomial(1, 0.3, 1000)
axes[0,2].hist(bernoulli, bins=3, color='orange', alpha=0.7)
axes[0,2].set_title('Bernoulli Distribution (p=0.3)')
axes[0,2].set_xlabel('ML use: binary classification')

# 4. Binomial Distribution — count of successes in n trials
# ML use: modelling counts
binomial = np.random.binomial(20, 0.5, 1000)
axes[1,0].hist(binomial, bins=20, color='purple', alpha=0.7)
axes[1,0].set_title('Binomial Distribution (n=20, p=0.5)')
axes[1,0].set_xlabel('ML use: count modelling')

# 5. Exponential Distribution — time between events
# ML use: modelling wait times, anomaly detection
exponential = np.random.exponential(1, 1000)
axes[1,1].hist(exponential, bins=30, color='red', alpha=0.7)
axes[1,1].set_title('Exponential Distribution')
axes[1,1].set_xlabel('ML use: anomaly detection')

# 6. Poisson Distribution — count of rare events
# ML use: NLP word counts, fraud detection
poisson = np.random.poisson(3, 1000)
axes[1,2].hist(poisson, bins=15, color='brown', alpha=0.7)
axes[1,2].set_title('Poisson Distribution')
axes[1,2].set_xlabel('ML use: word counts in NLP')

plt.suptitle('Common Probability Distributions in ML', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Statistics — Mean, Variance, Standard Deviation

These describe the shape of your data. Always compute these before training any ML model.

In [ ]:
np.random.seed(42)
data = np.random.normal(loc=50, scale=10, size=100)  # mean=50, std=10

# Mean — centre of the data
mean = np.mean(data)
print('Mean:', round(mean, 2))

# Median — middle value (robust to outliers)
median = np.median(data)
print('Median:', round(median, 2))

# Variance — average squared distance from mean
# High variance = spread out data
variance = np.var(data)
print('Variance:', round(variance, 2))

# Standard deviation — square root of variance (same units as data)
std = np.std(data)
print('Std Dev:', round(std, 2))

# What percentage of data falls within 1, 2, 3 standard deviations?
within_1std = np.sum(np.abs(data - mean) < std) / len(data)
within_2std = np.sum(np.abs(data - mean) < 2*std) / len(data)
within_3std = np.sum(np.abs(data - mean) < 3*std) / len(data)

print(f'\nWithin 1 std: {within_1std*100:.1f}%  (expected ~68%)')
print(f'Within 2 std: {within_2std*100:.1f}%  (expected ~95%)')
print(f'Within 3 std: {within_3std*100:.1f}%  (expected ~99.7%)')

## 6. Covariance and Correlation

Covariance measures how two features change together. Correlation is covariance normalised to be between -1 and 1.

In ML, highly correlated features are redundant — this is why we use PCA or drop features.

In [ ]:
np.random.seed(42)

# Generate correlated features
x1 = np.random.randn(100)
x2 = x1 * 2 + np.random.randn(100) * 0.5   # strongly correlated with x1
x3 = np.random.randn(100)                   # independent of x1

# Covariance matrix
data = np.stack([x1, x2, x3], axis=1)
cov_matrix = np.cov(data.T)
print('Covariance matrix:\n', cov_matrix.round(2))

# Correlation matrix (normalised covariance)
corr_matrix = np.corrcoef(data.T)
print('\nCorrelation matrix:\n', corr_matrix.round(2))
print('\nx1 and x2 correlation:', corr_matrix[0,1].round(2),  '(strong)')
print('x1 and x3 correlation:', corr_matrix[0,2].round(2), '(weak)')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(x1, x2, alpha=0.5)
axes[0].set_title(f'x1 vs x2 — corr = {corr_matrix[0,1]:.2f}')
axes[1].scatter(x1, x3, alpha=0.5)
axes[1].set_title(f'x1 vs x3 — corr = {corr_matrix[0,2]:.2f}')
plt.tight_layout()
plt.show()

## 7. Central Limit Theorem

No matter what distribution your data comes from, the distribution of sample means will approach a normal distribution as sample size increases.

Why ML cares: this is why we can assume normality in many ML scenarios, and why averaging (ensemble methods, mini-batch gradient descent) works so well.

In [ ]:
np.random.seed(42)

# Start with a skewed (non-normal) distribution
population = np.random.exponential(scale=2, size=100000)

# Take many samples and compute their means
sample_means_10  = [np.mean(np.random.choice(population, 10))   for _ in range(1000)]
sample_means_30  = [np.mean(np.random.choice(population, 30))   for _ in range(1000)]
sample_means_100 = [np.mean(np.random.choice(population, 100))  for _ in range(1000)]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].hist(population[:1000], bins=40, color='red', alpha=0.7)
axes[0].set_title('Original Distribution\n(Exponential — skewed)')

for ax, means, n, color in zip(axes[1:],
                                [sample_means_10, sample_means_30, sample_means_100],
                                [10, 30, 100],
                                ['orange', 'green', 'blue']):
    ax.hist(means, bins=40, color=color, alpha=0.7)
    ax.set_title(f'Sample Means (n={n})\nLooks more normal')

plt.suptitle('Central Limit Theorem — sample means become normal regardless of original distribution')
plt.tight_layout()
plt.show()

## 8. Entropy and Information

Entropy measures uncertainty or disorder in a distribution. In ML it shows up in decision trees (information gain) and cross-entropy loss (classification loss function).

**Formula:** H = -sum(p * log2(p))

High entropy = high uncertainty. Low entropy = low uncertainty.

In [ ]:
def entropy(probabilities):
    probabilities = np.array(probabilities)
    # avoid log(0) — replace 0s with tiny number
    probabilities = probabilities[probabilities > 0]
    return -np.sum(probabilities * np.log2(probabilities))

# Perfectly uncertain — all classes equally likely
print('Equal probs [0.5, 0.5]:', entropy([0.5, 0.5]))       # max entropy = 1.0

# Perfectly certain — one class always wins
print('Certain [1.0, 0.0]:', entropy([1.0, 0.0]))            # min entropy = 0.0

# Somewhere in between
print('Mixed [0.8, 0.2]:', round(entropy([0.8, 0.2]), 4))    # low entropy
print('Mixed [0.6, 0.4]:', round(entropy([0.6, 0.4]), 4))    # higher entropy

# Cross-entropy loss — used in classification
# Measures difference between predicted probabilities and true labels
def cross_entropy_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-10, 1)  # avoid log(0)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

y_true = np.array([1, 0, 1, 1, 0])

# Good predictions
y_good = np.array([0.9, 0.1, 0.8, 0.9, 0.1])
print('\nCross-entropy (good predictions):', round(cross_entropy_loss(y_true, y_good), 4))

# Bad predictions
y_bad = np.array([0.1, 0.9, 0.2, 0.3, 0.8])
print('Cross-entropy (bad predictions):', round(cross_entropy_loss(y_true, y_bad), 4))

---

## Mini Project — Naive Bayes Classifier from Scratch

**Task:** Build a spam detector using Bayes theorem — no sklearn.

This is exactly what the Naive Bayes classifier does: uses Bayes theorem to compute P(spam | words in email).

In [ ]:
# Training data — emails and labels (1=spam, 0=not spam)
emails = [
    ('free money win prize', 1),
    ('free click here now', 1),
    ('win free cash prize', 1),
    ('limited offer free gift', 1),
    ('meeting tomorrow at 10', 0),
    ('please review the document', 0),
    ('project update deadline', 0),
    ('lunch tomorrow sounds good', 0),
    ('can we reschedule the call', 0),
]

# Split into text and labels
texts  = [e[0] for e in emails]
labels = [e[1] for e in emails]

In [ ]:
# Step 1 — compute prior probabilities
n_total = len(labels)
n_spam  = sum(labels)
n_ham   = n_total - n_spam

p_spam = n_spam / n_total
p_ham  = n_ham / n_total

print('P(spam):', p_spam)
print('P(ham):', p_ham)

In [ ]:
# Step 2 — count word frequencies per class
spam_words = {}
ham_words  = {}

for text, label in emails:
    for word in text.split():
        if label == 1:
            spam_words[word] = spam_words.get(word, 0) + 1
        else:
            ham_words[word]  = ham_words.get(word, 0) + 1

total_spam_words = sum(spam_words.values())
total_ham_words  = sum(ham_words.values())
vocab_size       = len(set(spam_words) | set(ham_words))

print('Spam word counts:', spam_words)
print('Ham word counts:', ham_words)

In [ ]:
# Step 3 — compute word likelihood with Laplace smoothing
# Laplace smoothing prevents P(word) = 0 for unseen words
def word_likelihood(word, word_counts, total_words, vocab_size):
    count = word_counts.get(word, 0)
    return (count + 1) / (total_words + vocab_size)  # +1 smoothing

# Step 4 — predict function
def predict(text):
    words = text.split()

    # Start with log probabilities to avoid underflow
    log_p_spam = np.log(p_spam)
    log_p_ham  = np.log(p_ham)

    for word in words:
        log_p_spam += np.log(word_likelihood(word, spam_words, total_spam_words, vocab_size))
        log_p_ham  += np.log(word_likelihood(word, ham_words,  total_ham_words,  vocab_size))

    return 'SPAM' if log_p_spam > log_p_ham else 'HAM'

# Test
test_emails = [
    'free prize click now',
    'meeting at 10 tomorrow',
    'win free money now',
    'please send the document',
]

for email in test_emails:
    print(f'{predict(email):6} <- "{email}"')

---

## Common Mistakes to Avoid

1. **Confusing correlation with causation** — two features being correlated does not mean one causes the other. ML models find correlations, not causes.
2. **Ignoring class imbalance** — if 99% of your data is class 0, a model that always predicts 0 gets 99% accuracy. Use precision, recall, and F1 instead.
3. **Not checking data distribution** — always plot your data before training. Outliers and skewed distributions break many ML algorithms.
4. **Log of zero** — when computing probabilities, always use smoothing or clip values to avoid log(0) which is undefined.
5. **Assuming normal distribution** — not all data is normally distributed. Always verify before applying methods that assume normality.

---

## Interview Questions

1. What is the difference between probability and likelihood?
2. Explain Bayes theorem with a real ML example.
3. Why do we use log probabilities instead of raw probabilities in Naive Bayes?
4. What is entropy and how is it used in decision trees?
5. What is cross-entropy loss and why is it used for classification?
6. What does it mean for two features to be independent?
7. What is the Central Limit Theorem and why does it matter in ML?

---

**Next:** `03_calculus_gradient_descent.ipynb` — derivatives, chain rule, and how neural networks actually learn